
<div  style="text-align: center; line-height: 0; padding-top: 9px;">
  <img src="https://raw.githubusercontent.com/derar-alhussein/Databricks-Certified-Data-Engineer-Professional/main/Includes/images/bronze.png" width="60%">
</div>

In [0]:
%run ../Includes/Copy-Datasets

In [0]:
# Comment
files = dbutils.fs.ls(f"{bookstore.dataset_path}/kafka-raw")


In [0]:
df_raw = spark.read.json(f"{bookstore.dataset_path}/kafka-raw")
display(df_raw)

In [0]:
from pyspark.sql import functions as F

def process_bronze():
  
    schema = "key BINARY, value BINARY, topic STRING, partition LONG, offset LONG, timestamp LONG"

    query = (spark.readStream
                        .format("cloudFiles")
                        .option("cloudFiles.format", "json")
                        .schema(schema)
                        .load(f"{bookstore.dataset_path}/kafka-raw")
                        .withColumn("timestamp", (F.col("timestamp")/1000).cast("timestamp"))  
                        .withColumn("year_month", F.date_format("timestamp", "yyyy-MM"))
                  .writeStream
                      .option("checkpointLocation", f"{bookstore.checkpoint_path}/bronze")
                      .option("mergeSchema", True)
                      .partitionBy("topic", "year_month")
                      .trigger(availableNow=True)
                      .table("bronze"))
    
    query.awaitTermination()

In [0]:
process_bronze()

In [0]:
batch_df = spark.table("bronze")
display(batch_df)

In [0]:
%sql
SELECT * FROM bronze

In [0]:
%sql
SELECT DISTINCT(topic)
FROM bronze

In [0]:
bookstore.load_new_data()

In [0]:
process_bronze()

In [0]:
%sql
SELECT COUNT(*) FROM bronze